# CS103 ScienceWorld Assignment Tutorial

이 노트북은 학생들이 Jupyter Notebook에서 바로 ScienceWorld 과제를 실행할 수 있도록 만든 튜토리얼입니다.

다루는 내용:

- 패키지 설치
- Assignment 1 실행
- Assignment 2 실행
- 학생용 template를 상속해서 자기 agent를 만드는 시작점


## 1. 설치 전 준비

ScienceWorld는 Python뿐 아니라 Java도 필요합니다.

- Python 3.8+
- Java 8+

권장 순서:

1. 가상환경 생성
2. Jupyter 설치
3. `cs103-scienceworld` 설치

이미 수업용 환경이 준비되어 있다면 아래 셀부터 실행하면 됩니다.


In [ ]:
# Option A: 배포된 패키지를 설치하는 경우
%pip install cs103-scienceworld

# Option B: 저장소를 직접 clone해서 이 노트북을 examples/ 폴더에서 실행하는 경우
# %pip install -e ..


## 2. 과제용 API import

이번 과제용 helper는 `cs103_scienceworld.assignments` 아래에 들어 있습니다.
학생들은 보통 template agent를 상속해서 자신의 `choose_action()`만 채우면 됩니다.


In [ ]:
from cs103_scienceworld.assignments import (
    Assignment1PromptingTemplateAgent,
    Assignment1PromptingSolutionAgent,
    Assignment2RAGToolUseTemplateAgent,
    Assignment2RAGToolUseSolutionAgent,
    create_assignment_1_env,
    create_assignment_2_env,
    run_assignment_1_episode,
    run_assignment_2_episode,
)


## 3. Assignment 1: Prompting 빠르게 실행해 보기

먼저 정답 agent를 한 번 실행해서 전체 흐름을 확인합니다.

- `variation_idx=0`부터 `4`까지 총 5개 variation이 있습니다.
- `verbose=True`로 두면 각 step의 action과 observation이 출력됩니다.


In [ ]:
assignment1_result = run_assignment_1_episode(
    Assignment1PromptingSolutionAgent(),
    variation_idx=0,
    verbose=True,
)

assignment1_result


## 4. Assignment 1 환경 직접 들여다보기

학생 agent를 만들기 전에 task description, 초기 observation, valid actions를 직접 보는 것이 좋습니다.


In [ ]:
env = create_assignment_1_env(variation_idx=2)
observation, info = env.reset()

print("Task Description:")
print(info["taskDesc"])
print()
print("Initial Observation:")
print(observation)
print()
print("Some Valid Actions:")
print(env.get_valid_action_object_combinations()[:20])

env.close()


## 5. Assignment 1: 학생용 starter agent

아래 클래스는 template를 상속하는 최소 예시입니다.

학생이 실제로 채워야 하는 핵심은 `choose_action()`입니다.
이 메서드 안에서:

- prompt를 LLM에 보낼 수도 있고
- 규칙 기반으로 고를 수도 있고
- `valid_actions`에서 하나를 골라 반환해야 합니다.


In [ ]:
class MyAssignment1Agent(Assignment1PromptingTemplateAgent):
    def choose_action(self, valid_actions, prompt, observation, info):
        # TODO:
        # 1. prompt를 출력하거나 LLM으로 보내기
        # 2. valid_actions 중 하나를 고르기
        # 3. 반드시 문자열 action 하나를 return 하기
        
        # 임시 예시: 그냥 첫 번째 valid action 선택
        return valid_actions[0]


# 실행 예시
# my_assignment1_result = run_assignment_1_episode(MyAssignment1Agent(), variation_idx=1, verbose=True)
# my_assignment1_result


## 6. Assignment 2: RAG / Tool Use 빠르게 실행해 보기

Assignment 2는 recipe를 읽고 필요한 ingredient를 모아서 mix한 뒤, 결과물에 focus하는 task입니다.

정답 agent는 다음 흐름을 따릅니다.

1. kitchen에서 container 확보
2. recipe 읽기
3. recipe 내용 retrieval
4. ingredient 수집
5. mix
6. 결과물 focus


In [ ]:
assignment2_result = run_assignment_2_episode(
    Assignment2RAGToolUseSolutionAgent(),
    variation_idx=0,
    verbose=True,
)

assignment2_result


## 7. Assignment 2 환경 직접 확인하기

valid action space와 task description을 먼저 눈으로 보는 것이 좋습니다.


In [ ]:
env = create_assignment_2_env(variation_idx=0)
observation, info = env.reset()

print("Task Description:")
print(info["taskDesc"])
print()
print("Initial Observation:")
print(observation)
print()
print("Some Valid Actions:")
print(env.get_valid_action_object_combinations()[:25])

env.close()


## 8. Assignment 2: 학생용 starter agent

Assignment 2 template는 retrieval helper도 같이 제공합니다.

학생은 보통 아래 구조로 작업하면 됩니다.

- recipe를 읽은 뒤 retriever에 저장
- 필요한 ingredient를 query로 다시 retrieval
- valid action 중에서 현재 단계에 맞는 행동을 선택


In [ ]:
class MyAssignment2Agent(Assignment2RAGToolUseTemplateAgent):
    def choose_action(self, valid_actions, prompt, observation, info):
        # TODO:
        # 1. recipe observation을 보고 retriever에 저장
        # 2. 필요한 ingredient를 retrieval
        # 3. valid_actions에서 현재 단계에 맞는 action을 하나 선택
        
        # 임시 예시: 그냥 첫 번째 valid action 선택
        return valid_actions[0]


# 실행 예시
# my_assignment2_result = run_assignment_2_episode(MyAssignment2Agent(), variation_idx=1, verbose=True)
# my_assignment2_result


## 9. 자주 발생하는 문제

### import는 되는데 task가 안 열리는 경우

원인 후보:

- Java가 설치되지 않음
- 수업용 패키지가 최신이 아님
- 로컬 소스 수정 후 JAR rebuild를 안 함

### 로컬 저장소 버전을 쓰는 경우

Scala task를 수정했다면 Python 패키지만 다시 설치해서는 안 됩니다.
반드시 simulator JAR를 다시 빌드해야 합니다.

대략적인 순서:

1. `simulator/package.sh` 실행
2. `pip install -e ..` 또는 `pip install -e .`
3. notebook kernel 재시작

### 다음 추천 실험

- Assignment 1에서 prompt 형식을 바꿔 보기
- Assignment 2에서 retriever를 lexical overlap 대신 embedding 기반으로 바꿔 보기
- variation을 0부터 4까지 모두 돌려 보기
